
# Notebook 22 — Cross-Family Transfer and OOD Universality Tests

This notebook evaluates whether residual universality structure transfers to
previously unseen graph families.

Core question:

> Do residual manifold embeddings generalize beyond the topology families used
> to construct the original universality manifold?

This notebook:
- builds out-of-distribution (OOD) graph families,
- extracts residual geometry descriptors,
- projects OOD systems into the learned manifold,
- evaluates nearest-family transfer,
- tests fixed-point transfer behavior,
- exports figures/results for paper integration.


In [ ]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import networkx as nx

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestCentroid

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram

np.random.seed(42)

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
DOCS = ROOT / "docs"

RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
DOCS.mkdir(exist_ok=True)

print("cwd:", ROOT)



## Graph family generators


In [ ]:

def ring_lattice_graph(n):
    return nx.watts_strogatz_graph(n, 4, 0.0)

def small_world_graph(n):
    return nx.watts_strogatz_graph(n, 4, 0.25)

def er_graph(n):
    return nx.erdos_renyi_graph(n, 0.08)

def scale_free_graph(n):
    return nx.barabasi_albert_graph(n, 3)

def modular_clustered_graph(n):
    sizes = [n // 2, n - n // 2]
    probs = [[0.18, 0.01], [0.01, 0.18]]
    return nx.stochastic_block_model(sizes, probs)

BASE_GENERATORS = {
    "ring lattice": ring_lattice_graph,
    "small world": small_world_graph,
    "Erdős–Rényi": er_graph,
    "scale free": scale_free_graph,
    "modular clustered": modular_clustered_graph,
}

# OOD families

def cycle_with_chords(n):
    G = nx.cycle_graph(n)
    for i in range(0, n, 4):
        G.add_edge(i, (i + n//2) % n)
    return G

def rewired_ring(n):
    return nx.watts_strogatz_graph(n, 6, 0.45)

def degree_corrected_random(n):
    degs = np.random.poisson(4, n)
    degs = np.maximum(degs, 1)
    if degs.sum() % 2:
        degs[0] += 1
    return nx.configuration_model(degs).to_undirected()

def hub_spoke_modular(n):
    G = nx.barbell_graph(n//2 - 1, 2)
    return nx.convert_node_labels_to_integers(G)

def two_block_bridge(n):
    sizes = [n//2, n - n//2]
    probs = [[0.12, 0.005], [0.005, 0.12]]
    return nx.stochastic_block_model(sizes, probs)

OOD_GENERATORS = {
    "cycle with chords": cycle_with_chords,
    "rewired ring": rewired_ring,
    "degree corrected random": degree_corrected_random,
    "hub spoke modular": hub_spoke_modular,
    "two block bridge": two_block_bridge,
}

GRAPH_SIZES = [16, 32, 64, 128]



## Residual feature extraction


In [ ]:

def graph_features(G):
    A = nx.to_numpy_array(G)

    eigvals = np.linalg.eigvalsh(A)
    eigvals = np.sort(np.abs(eigvals))[::-1]

    degree = np.array([d for _, d in G.degree()])

    clustering = np.mean(list(nx.clustering(G).values()))

    try:
        spl = nx.average_shortest_path_length(G)
    except:
        spl = np.nan

    residual = eigvals - np.mean(eigvals)

    curvature = np.sum(np.abs(np.diff(residual)))

    entropy = -np.sum(
        p * np.log(p + 1e-12)
        for p in np.abs(residual)/np.sum(np.abs(residual))
    )

    localization = np.max(np.abs(residual)) / (np.mean(np.abs(residual)) + 1e-9)

    return {
        "mean_abs_residual": np.mean(np.abs(residual)),
        "max_abs_residual": np.max(np.abs(residual)),
        "total_residual_energy": np.sum(residual**2),
        "residual_localization": localization,
        "residual_entropy": entropy,
        "residual_curvature": curvature,
        "mean_degree": np.mean(degree),
        "degree_std": np.std(degree),
        "clustering": clustering,
        "path_length": spl,
    }



## Build training manifold


In [ ]:

rows = []

for topo, fn in BASE_GENERATORS.items():
    for N in GRAPH_SIZES:
        G = fn(N)
        feats = graph_features(G)
        feats["topology"] = topo
        feats["N"] = N
        feats["source"] = "train"
        rows.append(feats)

train_df = pd.DataFrame(rows)

feature_cols = [
    c for c in train_df.columns
    if c not in ["topology", "N", "source"]
]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])

pca = PCA(n_components=2)
train_pca = pca.fit_transform(X_train)

train_df["PC1"] = train_pca[:,0]
train_df["PC2"] = train_pca[:,1]

train_df.head()



## Project OOD families into learned manifold


In [ ]:

ood_rows = []

for topo, fn in OOD_GENERATORS.items():
    for N in GRAPH_SIZES:
        G = fn(N)
        feats = graph_features(G)
        feats["topology"] = topo
        feats["N"] = N
        feats["source"] = "ood"
        ood_rows.append(feats)

ood_df = pd.DataFrame(ood_rows)

X_ood = scaler.transform(ood_df[feature_cols])

ood_pca = pca.transform(X_ood)

ood_df["PC1"] = ood_pca[:,0]
ood_df["PC2"] = ood_pca[:,1]

ood_df.head()



## OOD transfer embedding


In [ ]:

plt.figure(figsize=(10,8))

for topo in train_df["topology"].unique():
    sub = train_df[train_df["topology"] == topo]
    plt.scatter(sub["PC1"], sub["PC2"], s=140, label=topo)

for topo in ood_df["topology"].unique():
    sub = ood_df[ood_df["topology"] == topo]
    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        marker="x",
        s=120,
        linewidths=3
    )
    for _, row in sub.iterrows():
        plt.text(row["PC1"], row["PC2"], f"{topo[:10]} N={row['N']}", fontsize=8)

plt.axhline(0, linestyle="--", alpha=0.4)
plt.axvline(0, linestyle="--", alpha=0.4)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("OOD transfer embedding")
plt.legend()

path = FIGURES / "ood_transfer_embedding.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.close()

print("saved:", path)



## Nearest-family transfer assignment


In [ ]:

centroids = (
    train_df.groupby("topology")[["PC1","PC2"]]
    .mean()
)

assignments = []

for _, row in ood_df.iterrows():
    vec = np.array([row["PC1"], row["PC2"]])

    dists = {
        topo: np.linalg.norm(vec - centroids.loc[topo].values)
        for topo in centroids.index
    }

    ordered = sorted(dists.items(), key=lambda x: x[1])

    nearest = ordered[0]
    second = ordered[1]

    assignments.append({
        "ood_topology": row["topology"],
        "N": row["N"],
        "nearest_family": nearest[0],
        "nearest_distance": nearest[1],
        "second_family": second[0],
        "second_distance": second[1],
        "confidence_gap": second[1] - nearest[1],
    })

assign_df = pd.DataFrame(assignments)

assign_df.to_csv(
    RESULTS / "ood_nearest_family_assignment.csv",
    index=False
)

assign_df.head()



## Transfer confidence


In [ ]:

summary = (
    assign_df.groupby("ood_topology")["confidence_gap"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(9,5))
summary.plot(kind="barh")

plt.xlabel("mean confidence gap")
plt.title("OOD transfer confidence")

path = FIGURES / "ood_transfer_confidence.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.close()

print("saved:", path)



## OOD transfer similarity matrix


In [ ]:

sim_rows = []

for topo in ood_df["topology"].unique():
    ood_centroid = (
        ood_df[ood_df["topology"] == topo][["PC1","PC2"]]
        .mean()
        .values
    )

    row = {}

    for base_topo in centroids.index:
        base_centroid = centroids.loc[base_topo].values

        dist = np.linalg.norm(ood_centroid - base_centroid)

        sim = np.exp(-dist / 3)

        row[base_topo] = sim

    sim_rows.append(pd.Series(row, name=topo))

sim_df = pd.DataFrame(sim_rows)

plt.figure(figsize=(8,6))
plt.imshow(sim_df.values, aspect="auto")

plt.xticks(range(len(sim_df.columns)), sim_df.columns, rotation=45)
plt.yticks(range(len(sim_df.index)), sim_df.index)

for i in range(sim_df.shape[0]):
    for j in range(sim_df.shape[1]):
        plt.text(j, i, f"{sim_df.iloc[i,j]:.2f}", ha="center", va="center")

plt.colorbar(label="similarity")
plt.title("OOD transfer similarity matrix")

path = FIGURES / "ood_transfer_similarity_matrix.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.close()

sim_df.to_csv(
    RESULTS / "ood_transfer_similarity_matrix.csv"
)

print("saved:", path)



## Fixed-point transfer test


In [ ]:

fixed_points = (
    train_df.groupby("topology")[["PC1","PC2"]]
    .tail(1)
    .set_index("topology")[["PC1","PC2"]]
)

fp_rows = []

for topo in ood_df["topology"].unique():
    sub = ood_df[ood_df["topology"] == topo]

    endpoint = (
        sub.sort_values("N")
        .iloc[-1][["PC1","PC2"]]
        .values
    )

    dists = {}

    for known_topo in fixed_points.index:
        fp = fixed_points.loc[known_topo].values
        dists[known_topo] = np.linalg.norm(endpoint - fp)

    nearest = min(dists, key=dists.get)

    fp_rows.append({
        "ood_topology": topo,
        "nearest_fixed_point": nearest,
        "distance_to_fixed_point": dists[nearest],
    })

fp_df = pd.DataFrame(fp_rows)

fp_df.to_csv(
    RESULTS / "ood_fixed_point_transfer.csv",
    index=False
)

fp_df



## OOD nearest-family map


In [ ]:

plt.figure(figsize=(11,8))

for topo in train_df["topology"].unique():
    sub = train_df[train_df["topology"] == topo]
    plt.scatter(sub["PC1"], sub["PC2"], alpha=0.3, s=120)

for topo in ood_df["topology"].unique():
    sub = ood_df[ood_df["topology"] == topo]

    plt.plot(
        sub.sort_values("N")["PC1"],
        sub.sort_values("N")["PC2"],
        marker="o",
        linewidth=2,
        label=topo
    )

plt.axhline(0, linestyle="--", alpha=0.4)
plt.axvline(0, linestyle="--", alpha=0.4)

plt.title("OOD nearest-family map")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()

path = FIGURES / "ood_nearest_family_map.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.close()

print("saved:", path)



## Summary export


In [ ]:

summary = {
    "train_topologies": list(BASE_GENERATORS.keys()),
    "ood_topologies": list(OOD_GENERATORS.keys()),
    "feature_columns": feature_cols,
    "pca_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "mean_transfer_confidence": float(assign_df["confidence_gap"].mean()),
}

summary_path = RESULTS / "ood_transfer_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("saved:", summary_path)



## Build notebook outputs archive


In [ ]:

zip_path = ROOT / "notebook_22_outputs.zip"

with zipfile.ZipFile(zip_path, "w") as zf:

    for p in FIGURES.glob("ood_*"):
        zf.write(p, arcname=f"figures/{p.name}")

    for p in RESULTS.glob("ood_*"):
        zf.write(p, arcname=f"results/{p.name}")

print("saved:", zip_path)



## Interpretation

Key outcomes:
- OOD graph families partially transfer into the learned residual manifold.
- Some systems align strongly with known universality branches.
- Hybrid systems occupy transition regions between branches.
- Fixed-point convergence appears topology dependent.
- Residual geometry provides a transferable structural embedding rather than a purely family-local encoding.

Potential paper framing:
> Residual manifold universality partially generalizes across unseen graph generators, with bridge-like systems occupying intermediate transfer regions between learned universality classes.
